In [1]:
import os

In [2]:
%pwd

'e:\\WineQuality\\research'

In [3]:
os.chdir('../')

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path

In [5]:
from WineQuality.constants import *
from WineQuality.utils.common import read_yaml,create_directories


In [6]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):

                 self.config=read_yaml(config_filepath)
                 self.params=read_yaml(params_filepath)
                 self.schema=read_yaml(schema_filepath)                 
                 create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)->DataIngestionConfig:
            config=self.config.data_ingestion

            create_directories([config.root_dir])
            data_ingestion_config=DataIngestionConfig(
                    root_dir=config.root_dir,
                    source_URL=config.source_URL,
                    local_data_file=config.local_data_file,
                    unzip_dir=config.unzip_dir
            )                 
        
            return data_ingestion_config

In [7]:
import os 
import urllib.request as request
import zipfile
from WineQuality import logger
from WineQuality.utils.common import get_size


In [8]:
class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename,headers=request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info:\n{headers}")
        else:
            logger.info(f"file already exists of size :{get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path=self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file,'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            
        

In [9]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()

except Exception as e:
    raise e

FILE: config\config.yaml
TYPE: <class 'dict'>
CONTENT: {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://github.com/entbappy/Branching-tutorial/raw/master/winequality.zip', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}}
[2026-06-12 21:02:43,439:INFO:common:yaml file: config\config.yaml loaded successfully]
FILE: params.yaml
TYPE: <class 'dict'>
CONTENT: {'key': 'val'}
[2026-06-12 21:02:43,441:INFO:common:yaml file: params.yaml loaded successfully]
FILE: schema.yaml
TYPE: <class 'dict'>
CONTENT: {'key': 'val'}
[2026-06-12 21:02:43,444:INFO:common:yaml file: schema.yaml loaded successfully]
[2026-06-12 21:02:43,446:INFO:common:created directory at: artifacts]
[2026-06-12 21:02:43,448:INFO:common:created directory at: artifacts/data_ingestion]
[2026-06-12 21:02:45,465:INFO:4087490991:artifacts/data_ingestion/data.zip download! with following info:
Connection: close
Content-Le